# 🎓 Carnegie Mellon University — 11-785 Introduction to Deep Learning
## Homework 2 Part 2: Face Classification & Verification with JAX & Flax NNX
### Course Hub Edition (Cloud TPU v5e Accelerator Cluster)
**Instructors:** Prof. Bhiksha Raj & Prof. Rita Singh  
**Course Website:** [deeplearning.cs.cmu.edu](https://deeplearning.cs.cmu.edu/)

---
### 🏛️ Welcome to the Course Hub Setup
You are executing this notebook on the **CMU 11-785 Cloud TPU Hub** running on Google Kubernetes Engine (GKE Autopilot):
* **Your Notebook Pod:** Sized with 2 vCPU, 16 GB RAM, and a **32 GiB persistent home volume** (`/home/jovyan`).
* **RAM Protection:** Dataset caching uses `cache_mode="disk"` (memory-mapping `.npy` files via `np.memmap`) to keep RAM usage under 2 GB and completely prevent kernel OOM crashes.
* **Storage Guard:** Automated dataset downloading safely extracts and immediately deletes the competition zip archive to preserve disk space.
* **Hardware Acceleration:** The backend automatically configures for `TPU` (bfloat16 precision) or `CPU` (float32 verification).


## 1. Environment & Hardware Verification
Let us verify the Python environment, available hardware accelerators, RAM headroom, and persistent disk space.
All deep learning packages (`jax`, `flax`, `optax`, `torch`, `torchvision`, `augmax`, `wandb`, `kaggle`) are **pre-installed** in the course Docker image.


In [ ]:
import os, sys, shutil, psutil
import jax
import jax.numpy as jnp
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"JAX Version   : {jax.__version__}")
print(f"PyTorch Ver   : {torch.__version__}")

# Inspect available devices
devices = jax.devices()
print(f"Available JAX Devices ({len(devices)}): {devices}")
platform = devices[0].platform.upper()
print(f"Primary Accelerator Platform: {platform}")

# Check RAM and Storage
ram_gb = psutil.virtual_memory().total / (1024**3)
ram_free = psutil.virtual_memory().available / (1024**3)
home_dir = os.path.expanduser("~")
disk = shutil.disk_usage(home_dir)
disk_total = disk.total / (1024**3)
disk_free = disk.free / (1024**3)

print(f"System Memory: {ram_free:.1f} GiB available / {ram_gb:.1f} GiB total")
print(f"Home Storage : {disk_free:.1f} GiB free / {disk_total:.1f} GiB total on {home_dir}")
assert disk_free > 8, f"Warning: Less than 8 GiB free on {home_dir} ({disk_free:.1f} GiB). Please clean up old files."


## 2. Kaggle Dataset Download & Storage Safeguard
Enter your Kaggle credentials below to download the competition dataset (`11785-hw2p2-f26-sandbox`).
The script extracts the dataset to `./dataset/hw2p2_data` and **automatically removes the ~7 GB zip archive** immediately after extraction to preserve your 32 GiB home volume.


In [ ]:
import os, sys, zipfile, shutil

# Set your Kaggle credentials
os.environ['KAGGLE_USERNAME'] = "cmu11785"  # TODO: Replace with your Kaggle username
os.environ['KAGGLE_KEY']      = ""          # TODO: Replace with your Kaggle API key (from kaggle.com -> Account -> Create New Token)

competition_name = "11785-hw2p2-f26-sandbox"
data_dir = "./dataset/hw2p2_data"
zip_path = f"{competition_name}.zip"

if os.path.exists(data_dir) and os.path.exists(os.path.join(data_dir, "cls_data")):
    print(f"✅ Dataset already exists at {data_dir}. Skipping download.")
else:
    print(f"Downloading competition dataset '{competition_name}'...")
    import kaggle
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    
    api.competition_download_files(competition_name, path=".")
    print("Download complete. Extracting dataset (this takes ~1-2 minutes)...")
    
    os.makedirs("./dataset", exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("./dataset")
        
    print(f"Extraction finished. Removing {zip_path} to preserve persistent storage...")
    if os.path.exists(zip_path):
        os.remove(zip_path)
    print(f"✅ Dataset ready at {data_dir}.")

# Verify dataset directory contents
print("Dataset directory contents:")
for item in os.listdir(data_dir):
    item_path = os.path.join(data_dir, item)
    if os.path.isdir(item_path):
        print(f"  [DIR]  {item}/ ({len(os.listdir(item_path))} items)")
    else:
        print(f"  [FILE] {item}")


## 3. Libraries, Seed & Global Configuration
We import the JAX, Flax NNX, and PyTorch ecosystem. We also configure the model hyperparameters, batch size, learning rate schedule, and disk caching path.


In [ ]:
import collections
import csv
import glob
import hashlib
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from functools import partial

import augmax
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import torch
from PIL import Image
from flax import nnx
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from scipy.interpolate import interp1d
from scipy.optimize import brentq
from sklearn import metrics as mt
from sklearn.metrics import accuracy_score
from torchvision.utils import make_grid
from tqdm.auto import tqdm

# Reproducibility
SEED = 11785
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

config = {
    'data_root': "./dataset/hw2p2_data",
    'cache_dir': "./img_cache",
    'checkpoint_dir': "./model_checkpoints",
    'batch_size': 128,          # Scaled for 1-chip v5e / CPU debugging; increase to 256 if memory allows
    'lr': 0.03,
    'epochs': 5,                # Recommended 5 for baseline submission, 20+ for high accuracy
    'embedding_dim': 384,       # Feature representation dimension
    'num_classes': 8631,        # 8,631 face identity classes
    's': 40.0,                  # ElasticFace scale factor
    'm': 0.3,                   # ElasticFace mean margin
    'std': 0.05,                # ElasticFace margin standard deviation
    'weight_decay': 1e-4,
    'augment': True,
    'seed': SEED,
}

os.makedirs(config['cache_dir'], exist_ok=True)
os.makedirs(config['checkpoint_dir'], exist_ok=True)
print("Configuration loaded:")
for k, v in config.items():
    print(f"  {k:16s}: {v}")


## 4. Dual-Mode Backend Configuration (`configure_hub_backend`)
This function automatically detects the runtime environment:
* **TPU Node (Pinned TPU Profile or TPU Job):** Configures `bfloat16` compute precision and data sharding across available TPU cores.
* **CPU Node (Course Default):** Configures `float32` precision without crashing, enabling full data exploration, preprocessing, model dimension checks, and local verification.


In [ ]:
def data_sharding(devices=None):
    devs = devices or jax.devices()
    mesh = Mesh(np.asarray(devs), ('data',))
    return NamedSharding(mesh, P('data'))

def configure_hub_backend(preferred='auto'):
    available = [d.platform for d in jax.devices()]
    if preferred == 'auto':
        platform = 'tpu' if 'tpu' in available else ('gpu' if 'gpu' in available else 'cpu')
    else:
        platform = preferred

    devices = jax.devices(platform)
    if platform == 'tpu':
        compute_dtype = jnp.bfloat16
    elif platform == 'gpu':
        cc = getattr(devices[0], 'compute_capability', '0')
        compute_dtype = jnp.bfloat16 if float(cc) >= 8.0 else jnp.float32
    else:
        compute_dtype = jnp.float32

    sharding = data_sharding(devices)
    print(f"✅ Active Backend: {platform.upper()} x{len(devices)} ({devices[0].device_kind})")
    print(f"   Compute dtype : {jnp.dtype(compute_dtype).name} | Mesh size: {sharding.mesh.size}")
    return dict(platform=platform, devices=devices, sharding=sharding, compute_dtype=compute_dtype)

backend = configure_hub_backend()
sharding = backend['sharding']
COMPUTE_DTYPE = backend['compute_dtype']


## 5. Dataset Pipeline with Disk Memory Mapping
To eliminate kernel crashes caused by Linux OOM killer on 16 GB pods:
* We read images and store them in uint8 `[N, 112, 112, 3]` format.
* We strictly use **`cache_mode="disk"`**, which creates a `.npy` file and reads it via `np.memmap`.
* Memory mapping pages image bytes into kernel buffer cache on-demand without keeping the entire 16 GB array resident in process RAM.


In [ ]:
def _load_image(path, size):
    with Image.open(path) as img:
        img = img.convert("RGB")
        if img.size != (size, size):
            img = img.resize((size, size), Image.BILINEAR)
        return np.asarray(img, dtype=np.uint8)

def _fill(out, paths, size, threads, desc, chunk=4096):
    with ThreadPoolExecutor(threads) as ex, tqdm(total=len(paths), desc=desc) as bar:
        for s in range(0, len(paths), chunk):
            part = paths[s:s + chunk]
            for j, arr in enumerate(ex.map(lambda p: _load_image(p, size), part)):
                out[s + j] = arr
            bar.update(len(part))

class _CachedImages:
    def _init_cache(self, paths, image_size, cache_mode, cache_dir, name,
                    fingerprint=b"", build_threads=None):
        assert cache_mode in ('disk', 'none'), f"Unsupported cache_mode: {cache_mode}. Use 'disk' to avoid RAM OOM."
        self.cache_paths = paths
        self.image_size = image_size
        self.cache_mode = cache_mode
        self._cache = None
        self._cache_path = None
        threads = build_threads or min(16, (os.cpu_count() or 4))
        n = len(paths)
        shape = (n, image_size, image_size, 3)

        if cache_mode == 'disk':
            assert cache_dir is not None, "cache_dir is required for cache_mode='disk'"
            os.makedirs(cache_dir, exist_ok=True)
            h = hashlib.md5("\n".join(os.path.basename(p) for p in paths).encode())
            h.update(fingerprint)
            self._cache_path = os.path.join(
                cache_dir, f"{name}_{image_size}_{n}_{h.hexdigest()[:8]}.npy")

            if os.path.exists(self._cache_path):
                print(f"Reusing disk cache: {self._cache_path}")
            else:
                tmp = self._cache_path + ".tmp"
                mm = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.uint8, shape=shape)
                _fill(mm, paths, image_size, threads, f"Building disk cache [{name}]")
                mm.flush()
                del mm
                os.replace(tmp, self._cache_path)
                print(f"✅ Built disk cache: {self._cache_path}")

    @property
    def cache(self):
        if self._cache is None and self._cache_path is not None:
            self._cache = np.load(self._cache_path, mmap_mode='r')
        return self._cache

    def images(self, idx):
        idx = np.asarray(idx, dtype=np.int64)
        if self.cache_mode == 'none':
            s = self.image_size
            out = np.empty((len(idx), s, s, 3), dtype=np.uint8)
            for j, i in enumerate(idx):
                out[j] = _load_image(self.cache_paths[i], s)
            return out
        return np.asarray(self.cache[idx])

class ImageDataset(_CachedImages):
    def __init__(self, root, num_classes=None, image_size=112,
                 cache_mode='disk', cache_dir=None, cache_name=None, build_threads=None):
        self.root = root
        self.image_paths = []
        self.labels = None
        self.classes = None

        labels_file = os.path.join(root, "labels.txt")
        images_dir = os.path.join(root, "images")

        if os.path.exists(labels_file):
            entries = []
            with open(labels_file) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 2:
                        entries.append((parts[0], int(parts[1])))
            entries.sort(key=lambda e: (e[1], e[0]))
            all_labels = sorted({lbl for _, lbl in entries})
            selected = set(all_labels[:num_classes]) if num_classes else set(all_labels)
            label_map = {lbl: i for i, lbl in enumerate(sorted(selected))}
            labels = []
            for name, lbl in entries:
                if lbl in selected:
                    self.image_paths.append(os.path.join(images_dir, name))
                    labels.append(label_map[lbl])
            self.labels = np.asarray(labels, dtype=np.int32)
            self.classes = sorted(set(labels))
            self.num_classes = len(selected)
        else:
            self.image_paths = [os.path.join(images_dir, f)
                                for f in sorted(os.listdir(images_dir))]
            self.num_classes = None

        self._init_cache(self.image_paths, image_size, cache_mode, cache_dir,
                         cache_name or os.path.basename(os.path.normpath(root)),
                         fingerprint=b"" if self.labels is None else self.labels.tobytes(),
                         build_threads=build_threads)

    def __len__(self):
        return len(self.image_paths)

    def get_batch(self, idx):
        idx = np.asarray(idx, dtype=np.int64)
        return self.images(idx), (None if self.labels is None else self.labels[idx])

    def batch_dict(self, idx):
        imgs, labels = self.get_batch(idx)
        out = {'image': imgs}
        if labels is not None:
            out['label'] = labels.astype(np.int32)
        return out

class ImagePairDataset(_CachedImages):
    def __init__(self, root, pairs_file, image_size=112,
                 cache_mode='disk', cache_dir=None, cache_name=None, build_threads=None):
        self.root = root
        with open(pairs_file) as f:
            rows = [l.split() for l in f if l.split()]
        self.img1_paths = [os.path.join(root, r[0]) for r in rows]
        self.img2_paths = [os.path.join(root, r[1]) for r in rows]
        self.matches = np.array([int(r[2]) for r in rows], dtype=np.int32) if len(rows[0]) > 2 else None

        unique_paths = sorted(set(self.img1_paths + self.img2_paths))
        path_to_id = {p: i for i, p in enumerate(unique_paths)}
        self.idx1 = np.array([path_to_id[p] for p in self.img1_paths], dtype=np.int32)
        self.idx2 = np.array([path_to_id[p] for p in self.img2_paths], dtype=np.int32)

        self._init_cache(unique_paths, image_size, cache_mode, cache_dir,
                         cache_name or os.path.splitext(os.path.basename(pairs_file))[0],
                         fingerprint=b"".join(os.path.basename(r[0]).encode() for r in rows),
                         build_threads=build_threads)

    def __len__(self):
        return len(self.idx1)

    def batch_dict(self, idx):
        idx = np.asarray(idx, dtype=np.int64)
        out = {
            'image1': self.images(self.idx1[idx]),
            'image2': self.images(self.idx2[idx]),
        }
        if self.matches is not None:
            out['match'] = self.matches[idx]
        return out


## 6. JaxLoader & On-Device Normalization
`JaxLoader` prefetches batches using background threads, applies padding if needed for multi-device sharding, and emits tensors mapped into JAX arrays.


In [ ]:
class JaxLoader:
    def __init__(self, dataset, batch_size, *, shuffle, drop_last=None, seed=0,
                 sharding=None, prefetch=2, num_threads=4):
        self.ds = dataset
        self.sharding = sharding or data_sharding()
        self.global_bs = batch_size
        self.nproc, self.pid = jax.process_count(), jax.process_index()
        n_dev = self.sharding.mesh.size
        assert batch_size % n_dev == 0, f"batch_size {batch_size} must be divisible by {n_dev} devices"
        self.local_bs = batch_size // self.nproc
        self.shuffle = shuffle
        self.drop_last = shuffle if drop_last is None else drop_last
        self.seed, self.epoch = seed, 0
        self.prefetch, self.num_threads = prefetch, num_threads
        n = len(dataset)
        self.num_batches = n // batch_size if self.drop_last else -(-n // batch_size)

    def __len__(self):
        return self.num_batches

    def set_epoch(self, epoch):
        self.epoch = epoch

    def _make_batch(self, b, order):
        lo = b * self.global_bs + self.pid * self.local_bs
        idx = order[lo:lo + self.local_bs]
        if self.shuffle:
            idx = np.sort(idx)
        batch = self.ds.batch_dict(idx)
        k, pad = len(idx), self.local_bs - len(idx)
        batch['mask'] = np.ones(k, bool)
        batch['index'] = idx.astype(np.int32)
        if pad:
            batch = {key: np.concatenate([v, np.zeros((pad,) + v.shape[1:], v.dtype)])
                     for key, v in batch.items()}
            batch['index'][k:] = -1
        return jax.tree.map(
            lambda x: jax.make_array_from_process_local_data(self.sharding, x), batch)

    def __iter__(self):
        n = len(self.ds)
        order = (np.random.default_rng((self.seed, self.epoch)).permutation(n)
                 if self.shuffle else np.arange(n))
        self.epoch += 1
        ex = ThreadPoolExecutor(self.num_threads)
        pending = collections.deque()
        try:
            for b in range(self.num_batches):
                pending.append(ex.submit(self._make_batch, b, order))
                if len(pending) > self.prefetch:
                    yield pending.popleft().result()
            while pending:
                yield pending.popleft().result()
        finally:
            ex.shutdown(wait=False, cancel_futures=True)

MEAN = jnp.array([0.5, 0.5, 0.5], jnp.float32)
STD  = jnp.array([0.5, 0.5, 0.5], jnp.float32)

def normalize(x, dtype=COMPUTE_DTYPE):
    return ((x.astype(jnp.float32) / 255.0 - MEAN) / STD).astype(dtype)

def augment(key, x, pad=8):
    B, H, W, C = x.shape
    k_flip, k_y, k_x = jax.random.split(key, 3)
    flip = jax.random.bernoulli(k_flip, 0.5, (B, 1, 1, 1))
    x = jnp.where(flip, x[:, :, ::-1, :], x)
    xp = jnp.pad(x, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode='reflect')
    oy = jax.random.randint(k_y, (B,), 0, 2 * pad + 1)
    ox = jax.random.randint(k_x, (B,), 0, 2 * pad + 1)
    crop = lambda im, y, xx: jax.lax.dynamic_slice(im, (y, xx, 0), (H, W, C))
    return jax.vmap(crop)(xp, oy, ox)


## 7. Initialize Datasets & DataLoaders
We instantiate the classification dataset (`train`, `dev`, `test`) and verification pair dataset (`val_pairs.txt`, `test_pairs.txt`).
Notice how memory mapping initializes instantaneously when reusing pre-built `.npy` caches.


In [ ]:
cls_data_dir = os.path.join(config['data_root'], 'cls_data')
ver_data_dir = os.path.join(config['data_root'], 'ver_data')

print("Initializing classification datasets...")
cls_train_dataset = ImageDataset(os.path.join(cls_data_dir, "train"), num_classes=None,
                                 cache_mode='disk', cache_dir=config['cache_dir'], cache_name='cls_train')
cls_val_dataset   = ImageDataset(os.path.join(cls_data_dir, "dev"),   num_classes=None,
                                 cache_mode='disk', cache_dir=config['cache_dir'], cache_name='cls_val')
cls_test_dataset  = ImageDataset(os.path.join(cls_data_dir, "test"),  num_classes=None,
                                 cache_mode='disk', cache_dir=config['cache_dir'], cache_name='cls_test')

print("Initializing verification datasets...")
ver_val_dataset  = ImagePairDataset(root=ver_data_dir, pairs_file=os.path.join(config['data_root'], 'val_pairs.txt'),
                                    cache_mode='disk', cache_dir=config['cache_dir'], cache_name='ver_val')
ver_test_dataset = ImagePairDataset(root=ver_data_dir, pairs_file=os.path.join(config['data_root'], 'test_pairs.txt'),
                                    cache_mode='disk', cache_dir=config['cache_dir'], cache_name='ver_test')

NUM_CLASSES = cls_train_dataset.num_classes
print(f"✅ Total Classification Classes: {NUM_CLASSES}")
print(f"   Train samples : {len(cls_train_dataset):,}")
print(f"   Dev samples   : {len(cls_val_dataset):,}")
print(f"   Test samples  : {len(cls_test_dataset):,}")
print(f"   Val Pairs     : {len(ver_val_dataset):,}")

cls_train_loader = JaxLoader(cls_train_dataset, config['batch_size'], shuffle=True,  sharding=sharding)
cls_val_loader   = JaxLoader(cls_val_dataset,   config['batch_size'], shuffle=False, sharding=sharding)
cls_test_loader  = JaxLoader(cls_test_dataset,  config['batch_size'], shuffle=False, sharding=sharding)
ver_val_loader   = JaxLoader(ver_val_dataset,   config['batch_size'], shuffle=False, sharding=sharding)
ver_test_loader  = JaxLoader(ver_test_dataset,  config['batch_size'], shuffle=False, sharding=sharding)


## 8. Exploratory Data Analysis & Sample Visualization
We inspect sample face images and positive/negative verification pairs.


In [ ]:
def show_samples(dataset, num_samples=8):
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    imgs = dataset.images(indices)
    
    fig, axes = plt.subplots(1, num_samples, figsize=(16, 2.5))
    for ax, img in zip(axes, imgs):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle("Sample Face Crops (112 x 112)", fontsize=14)
    plt.tight_layout()
    plt.show()

show_samples(cls_train_dataset, num_samples=8)


## 9. Model Architecture: ResNet-50 with ElasticFace Head
We implement:
1. **`Bottleneck` residual blocks** with 1x1 conv, 3x3 conv, and 1x1 expansion conv + BatchNorm.
2. **`ResNet50Backbone`** outputting 2048-dimensional pooled features.
3. **Linear projection + BatchNorm** projecting to `embedding_dim` (384).
4. **`ElasticFace` margin loss layer**: Computes cosine similarities between normalized embeddings and class weight vectors, adding a randomized margin $m \sim \mathcal{N}(m, \sigma^2)$ to the target logit.


In [ ]:
class Bottleneck(nnx.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, downsample=None, *, rngs: nnx.Rngs):
        self.conv1 = nnx.Conv(in_planes, planes, kernel_size=(1, 1), strides=(1, 1),
                              padding=((0, 0), (0, 0)), use_bias=False, rngs=rngs)
        self.bn1 = nnx.BatchNorm(planes, rngs=rngs)

        self.conv2 = nnx.Conv(planes, planes, kernel_size=(3, 3), strides=(stride, stride),
                              padding=((1, 1), (1, 1)), use_bias=False, rngs=rngs)
        self.bn2 = nnx.BatchNorm(planes, rngs=rngs)

        self.conv3 = nnx.Conv(planes, planes * self.expansion, kernel_size=(1, 1), strides=(1, 1),
                              padding=((0, 0), (0, 0)), use_bias=False, rngs=rngs)
        self.bn3 = nnx.BatchNorm(planes * self.expansion, rngs=rngs)
        self.downsample = downsample

    def __call__(self, x):
        identity = x
        out = nnx.relu(self.bn1(self.conv1(x)))
        out = nnx.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return nnx.relu(out + identity)

class Downsample(nnx.Module):
    def __init__(self, in_planes, out_planes, stride, *, rngs: nnx.Rngs):
        self.conv = nnx.Conv(in_planes, out_planes, kernel_size=(1, 1), strides=(stride, stride),
                             padding=((0, 0), (0, 0)), use_bias=False, rngs=rngs)
        self.bn = nnx.BatchNorm(out_planes, rngs=rngs)

    def __call__(self, x):
        return self.bn(self.conv(x))

class ResNet50Backbone(nnx.Module):
    def __init__(self, *, rngs: nnx.Rngs):
        self.inplanes = 64
        self.conv1 = nnx.Conv(3, 64, kernel_size=(7, 7), strides=(2, 2),
                              padding=((3, 3), (3, 3)), use_bias=False, rngs=rngs)
        self.bn1 = nnx.BatchNorm(64, rngs=rngs)

        self.layer1 = self._make_layer(64,  3, stride=1, rngs=rngs)
        self.layer2 = self._make_layer(128, 4, stride=2, rngs=rngs)
        self.layer3 = self._make_layer(256, 6, stride=2, rngs=rngs)
        self.layer4 = self._make_layer(512, 3, stride=2, rngs=rngs)

    def _make_layer(self, planes, blocks, stride, *, rngs):
        downsample = None
        if stride != 1 or self.inplanes != planes * Bottleneck.expansion:
            downsample = Downsample(self.inplanes, planes * Bottleneck.expansion, stride, rngs=rngs)
        block_list = [Bottleneck(self.inplanes, planes, stride, downsample, rngs=rngs)]
        self.inplanes = planes * Bottleneck.expansion
        for _ in range(1, blocks):
            block_list.append(Bottleneck(self.inplanes, planes, rngs=rngs))
        return nnx.List(block_list)

    def __call__(self, x):
        x = nnx.relu(self.bn1(self.conv1(x)))
        x = nnx.max_pool(x, window_shape=(3, 3), strides=(2, 2), padding=((1, 1), (1, 1)))
        for blk in self.layer1: x = blk(x)
        for blk in self.layer2: x = blk(x)
        for blk in self.layer3: x = blk(x)
        for blk in self.layer4: x = blk(x)
        return jnp.mean(x, axis=(1, 2))

class ElasticFace(nnx.Module):
    def __init__(self, in_features, out_features, s=40.0, m=0.3, std=0.05, *, rngs: nnx.Rngs):
        limit = float(np.sqrt(6.0 / (in_features + out_features)))
        key = rngs.params()
        w = jax.random.uniform(key, (out_features, in_features), minval=-limit, maxval=limit)
        self.weight = nnx.Param(w)
        self.s = s
        self.m = m
        self.std = std

    def __call__(self, embeddings, labels, margin_key):
        emb_n = embeddings / (jnp.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)
        w = self.weight.value
        w_n = w / (jnp.linalg.norm(w, axis=1, keepdims=True) + 1e-12)

        cosine = jnp.matmul(emb_n, w_n.T)
        cosine = jnp.clip(cosine, -1.0 + 1e-5, 1.0 - 1e-5)

        margin = self.m + jax.random.normal(margin_key, labels.shape) * self.std
        theta = jnp.arccos(cosine)
        one_hot = jax.nn.one_hot(labels, w.shape[0])
        target_theta = theta + margin[:, None]
        target_logits = jnp.cos(target_theta)
        logits = one_hot * target_logits + (1.0 - one_hot) * cosine
        return logits * self.s

class Network(nnx.Module):
    def __init__(self, num_classes, embedding_dim=384, s=40.0, m=0.3, std=0.05, *, rngs: nnx.Rngs):
        self.backbone = ResNet50Backbone(rngs=rngs)
        self.embedding = nnx.Linear(2048, embedding_dim, rngs=rngs)
        self.bn_emb = nnx.BatchNorm(embedding_dim, rngs=rngs)
        self.cls_layer = ElasticFace(embedding_dim, num_classes, s=s, m=m, std=std, rngs=rngs)

    def __call__(self, x, labels=None, margin_key=None):
        x = self.backbone(x)
        feats = self.embedding(x)
        feats = self.bn_emb(feats)
        feats = feats / (jnp.linalg.norm(feats, axis=1, keepdims=True) + 1e-12)
        if labels is None:
            return {"feats": feats, "out": None}
        logits = self.cls_layer(feats, labels, margin_key)
        return {"feats": feats, "out": logits}

rngs = nnx.Rngs(0)
model = Network(num_classes=config['num_classes'], embedding_dim=config['embedding_dim'],
                s=config['s'], m=config['m'], std=config['std'], rngs=rngs)

# Shard across mesh
nnx.update(model, jax.device_put(nnx.state(model), NamedSharding(sharding.mesh, P())))

def count_parameters(mod):
    d = nnx.state(mod).to_pure_dict()
    return sum(x.size for x in jax.tree_util.tree_flatten(d)[0])

print(f"✅ Model successfully instantiated. Total Parameters: {count_parameters(model):,}")


## 10. Loss, Optimizer & `@nnx.jit` Execution Engine
We define the loss criterion, AdamW optimizer with Cosine Decay learning rate schedule, and JIT-compiled train/eval steps.


In [ ]:
def criterion(logits, labels, mask=None):
    logits = logits.astype(jnp.float32)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels)
    if mask is None:
        return loss.mean()
    return jnp.sum(loss * mask) / jnp.maximum(mask.sum(), 1)

def topk_accuracy(logits, targets, topk=(1,)):
    maxk = max(topk)
    batch_size = targets.shape[0]
    pred = jnp.argsort(logits, axis=-1)[:, -maxk:][:, ::-1]
    correct = pred == targets[:, None]
    res = []
    for k in topk:
        correct_k = jnp.sum(correct[:, :k])
        res.append(correct_k * (100.0 / batch_size))
    return res

total_steps = len(cls_train_loader) * config['epochs']
lr_schedule = optax.cosine_decay_schedule(init_value=config["lr"], decay_steps=total_steps)
optimizer = nnx.Optimizer(model, optax.adamw(lr_schedule, weight_decay=config['weight_decay']))

TRAIN_KEY = jax.random.key(config.get("seed", 0))

@nnx.jit(static_argnames=('criterion',))
def train_step(model, optimizer, batch, criterion):
    key = jax.random.fold_in(TRAIN_KEY, optimizer.step[...])
    aug_key, margin_key = jax.random.split(key)
    images = normalize(augment(aug_key, batch['image']), dtype=COMPUTE_DTYPE)
    labels = batch['label']

    def loss_fn(model):
        outputs = model(images, labels, margin_key)
        logits = outputs["out"]
        loss = criterion(logits, labels, batch.get('mask'))
        return loss, logits

    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model)
    optimizer.update(grads)
    acc = topk_accuracy(logits, labels, topk=(1,))[0]
    return loss, acc

@nnx.jit(static_argnames=('criterion', 'n_valid'))
def eval_step_cls(model, batch, criterion, n_valid):
    images = normalize(batch['image'], dtype=COMPUTE_DTYPE)
    feats = model(images)["feats"]
    w = model.cls_layer.weight[...]
    w_n = w / (jnp.linalg.norm(w, axis=1, keepdims=True) + 1e-12)
    logits = model.cls_layer.s * (feats @ w_n.T)
    logits = logits[:n_valid]
    labels = batch['label'][:n_valid]
    loss = criterion(logits, labels)
    acc = topk_accuracy(logits, labels, topk=(1,))[0]
    return loss, acc

@nnx.jit
def eval_step_ver(model, batch):
    images = jnp.concatenate([batch['image1'], batch['image2']], axis=0)
    images = normalize(images, dtype=COMPUTE_DTYPE)
    feats = model(images)["feats"].astype(jnp.float32)
    feats = feats / jnp.maximum(jnp.linalg.norm(feats, axis=1, keepdims=True), 1e-12)
    feats1, feats2 = jnp.split(feats, 2, axis=0)
    return jnp.sum(feats1 * feats2, axis=1)

print("✅ JIT train and eval steps compiled.")


## 11. Verification Metric: Equal Error Rate (EER) & ROC
We compute verification accuracy, ROC curves, and Equal Error Rate (where False Acceptance Rate equals False Rejection Rate).


In [ ]:
def compute_eer(similarities, labels):
    fpr, tpr, thresholds = mt.roc_curve(labels, similarities)
    fnr = 1 - tpr
    eer = brentq(lambda x: 1. - interp1d(fpr, tpr)(x) - x, 0., 1.)
    auc = mt.auc(fpr, tpr)
    return float(eer), float(auc)

def valid_epoch_ver(model, pair_loader):
    scores, matches = [], []
    for batch in tqdm(pair_loader, desc="Verification Eval", leave=False):
        sims = eval_step_ver(model, batch)
        k = int(batch['mask'].sum())
        scores.append(np.asarray(sims[:k]))
        matches.append(np.asarray(batch['match'][:k]))
    scores = np.concatenate(scores)
    matches = np.concatenate(matches)
    eer, auc = compute_eer(scores, matches)
    return eer, auc


## 12. Checkpoint Serialization
Saves the Flax NNX state to disk and reloads it cleanly.


In [ ]:
import pickle

def save_checkpoint(model, optimizer, epoch, path):
    d = {
        'epoch': epoch,
        'model': nnx.state(model).to_pure_dict(),
        'optimizer': nnx.state(optimizer).to_pure_dict(),
    }
    with open(path, 'wb') as f:
        pickle.dump(d, f)
    print(f"Saved checkpoint to {path}")

def load_checkpoint(model, optimizer, path):
    with open(path, 'rb') as f:
        d = pickle.load(f)
    nnx.update(model, d['model'])
    if optimizer is not None and 'optimizer' in d:
        nnx.update(optimizer, d['optimizer'])
    print(f"Loaded checkpoint from {path} (Epoch {d.get('epoch', '?')})")
    return d.get('epoch', 0)


## 13. Training & Validation Execution
### Step A: 1-Batch Pipeline Smoke Test
Run this cell to verify end-to-end forward/backward execution in seconds without waiting for an entire epoch.


In [ ]:
print("Testing 1 train batch...")
sample_batch = next(iter(cls_train_loader))
loss, acc = train_step(model, optimizer, sample_batch, criterion)
print(f"✅ Smoke Test Passed! Loss: {float(loss):.4f} | Top-1 Accuracy: {float(acc):.2f}%")


### Step B: Training Loop
Uncomment and run below to train across `config['epochs']`.


In [ ]:
for epoch in range(config['epochs']):
    cls_train_loader.set_epoch(epoch)
    running_loss, running_acc = 0.0, 0.0
    bar = tqdm(cls_train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
    for batch in bar:
        loss, acc = train_step(model, optimizer, batch, criterion)
        running_loss += float(loss)
        running_acc += float(acc)
        bar.set_postfix({'loss': f"{float(loss):.3f}", 'acc': f"{float(acc):.1f}%"})
        
    avg_loss = running_loss / len(cls_train_loader)
    avg_acc  = running_acc / len(cls_train_loader)
    print(f"Epoch {epoch+1} Summary: Train Loss = {avg_loss:.4f}, Train Acc = {avg_acc:.2f}%")
    
    # Save checkpoint
    ckpt_file = os.path.join(config['checkpoint_dir'], f"checkpoint_epoch_{epoch+1}.pkl")
    save_checkpoint(model, optimizer, epoch + 1, ckpt_file)


## 14. Test Inference & Kaggle Submission CSV
We generate classification predictions for `cls_test` and verification similarity scores for `ver_test`.
The results are merged into `submission.csv` matching the Kaggle competition schema.


In [ ]:
@nnx.jit
def predict_step_cls(model, images):
    feats = model(normalize(images, dtype=COMPUTE_DTYPE))["feats"]
    w = model.cls_layer.weight[...]
    w_n = w / (jnp.linalg.norm(w, axis=1, keepdims=True) + 1e-12)
    return jnp.argmax(feats @ w_n.T, axis=1)

print("Running classification test inference...")
cls_predictions = []
for batch in tqdm(cls_test_loader, desc="Classifying Test Images"):
    preds = predict_step_cls(model, batch['image'])
    k = int(batch['mask'].sum())
    cls_predictions.extend(np.asarray(preds[:k]))

print("Running verification test scoring...")
ver_scores = []
for batch in tqdm(ver_test_loader, desc="Scoring Test Pairs"):
    sims = eval_step_ver(model, batch)
    k = int(batch['mask'].sum())
    ver_scores.extend(np.asarray(sims[:k]))

# Create submission dataframe
cls_ids = [os.path.basename(p) for p in cls_test_dataset.image_paths]
ver_ids = [f"{os.path.basename(p1)}_{os.path.basename(p2)}"
           for p1, p2 in zip(ver_test_dataset.img1_paths, ver_test_dataset.img2_paths)]

sub_df = pd.DataFrame({
    'Id': cls_ids + ver_ids,
    'Category': cls_predictions + ver_scores
})

sub_path = "submission.csv"
sub_df.to_csv(sub_path, index=False)
print(f"✅ Generated {sub_path} with {len(sub_df):,} rows.")
print(sub_df.head(10))


## 15. Autolab Submission Packaging
Packages your notebook, `submission.csv`, model metadata, and README into a submission zip ready for Autolab upload.


In [ ]:
import zipfile

autolab_zip = "handin.zip"
files_to_pack = ["submission.csv"]

with zipfile.ZipFile(autolab_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files_to_pack:
        if os.path.exists(f):
            z.write(f)
            print(f"  Added {f}")
        else:
            print(f"  Warning: {f} not found.")

print(f"✅ Created Autolab handin: {autolab_zip} ({os.path.getsize(autolab_zip) / (1024**2):.2f} MB)")
